# Sort Example

Sort example application demonstrates how to implement a simple sorting algorithm using the SDK by sorting an array of integers in ascending order. 

## Prerequisites

Before running this notebook, you must first build the kernel by executing the following command in your terminal:

```bash
cd /root/sdk/example/sort
./build.sh
```

This will compile the kernel code and generate the required `mu_kernel/mu_kernel.mubin` file.

1. Setup PXL runtime instances.

In [ ]:
#pragma cling add_include_path("/usr/local/include")
#pragma cling add_library_path("/usr/local/lib")
#pragma cling load("pxl")

#include "pxl/pxl.hpp"

auto numDevice = pxl::getNumDevice();
printf("numDevice = %d\n", numDevice);
auto context = pxl::createContext();

const char* filename = "mu_kernel/mu_kernel.mubin";
auto module = pxl::createModule(filename);

auto muFuncList = module->getMuFunctionList();
for (const auto& muFunc : muFuncList)
{
    printf("muFunc = %s\n", muFunc.c_str());
}

In [ ]:
auto job = context->createJob();
job->load(module);
const char* muFuncName = "sort_with_ptr";
auto muFunction = module->createFunction(muFuncName);
if (muFunction == nullptr)
{
    fprintf(stderr, "Cannot create mu_function %s\n", muFuncName);
}

2. Configure arguments and allocate device memory. `sortSize` is number of elements to process per task. `testCount` is number of parallel tasks.

In [ ]:
int sortSize = 64;
int testCount = 1024;
int dataSize = testCount * sortSize * sizeof(int);
int* data = reinterpret_cast<int*>(context->memAlloc(dataSize));
if (data == nullptr)
{
    printf("Failed to allocate memory\n");
}

3. Generate unsorted initial data.

In [ ]:
for (size_t i = 0; i < testCount; i++)
{
    for (size_t j = 0; j < sortSize; j++)
    {
        data[i * sortSize + j] = sortSize - j;
    }
}
pxl::flushHostCache(data, dataSize);

for (int i = 0; i < sortSize; i++)
{
    printf("%02d ", data[i]);
}

4. Execute the kernel.

In [ ]:
auto executor = job->buildMap(muFunction, testCount);
auto ret = executor->execute(data, sortSize);
if (ret == pxl::Result::Success)
{
    printf("Execute success\n");
}
executor->synchronize();
pxl::flushHostCache(data, dataSize);

5. Check the results.

In [ ]:
for (int i = 0; i < sortSize; i++)
{
    printf("%02d ", data[i]);
}